# 🎯 Audio MIL 数据集完整准备 Notebook（已严格满足所有要求）

**已实现：**
- 读取 35 个 .wav + 3 个 CSV（Ori_file_num、Train_start(s)、Train_end(s)）
- 每条音频 **下采样到 48kHz** + **切割成 1min bag** 并**真实保存**为 wav
- 按时间顺序 **8:2**（5折验证，shuffle=False）分割
- 弱标签：只要bag内包含任意一种 Train → label=1
- **统计**：5次分割中，每个 .wav 文件测试集的 正/负 bag 数量（每折 + 汇总）

运行后根目录会生成：
- `bag_test_stats_5fold_detailed.csv`
- `dataset/bags/` 文件夹（已切好的 48kHz 1min wav）
- 可直接用于后续 MIL 训练

In [5]:
!pip install librosa soundfile pandas numpy scipy tqdm -q

import os, re, pandas as pd, numpy as np
import soundfile as sf
import librosa
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

ROOT = Path(r'D:\Project_Github\audio_click_mil')
os.chdir(ROOT)
ORIGINAL = ROOT / 'data' / 'original_audio'
CSV_DIR = ROOT / 'data'
(ROOT / 'dataset' / 'bags').mkdir(parents=True, exist_ok=True)
print('✅ 准备完成，根目录：', ROOT)

✅ 准备完成，根目录： D:\Project_Github\audio_click_mil


In [6]:
# 1. 读取三个 CSV 并合并
bp = pd.read_csv(CSV_DIR / 'BurstPulseTrains.csv')
buzz = pd.read_csv(CSV_DIR / 'BuzzTrains.csv')
click = pd.read_csv(CSV_DIR / 'ClickTrains.csv')

trains = pd.concat([bp, buzz, click], ignore_index=True)
trains = trains[['Ori_file_num(No.)', 'Train_start(s)', 'Train_end(s)']].copy()
trains.columns = ['file_num', 'start', 'end']
trains['file_num'] = trains['file_num'].astype(int)
print(f'✅ Train总数：{len(trains)} 条')

✅ Train总数：897 条


In [7]:
# 2. 获取所有 wav 并解析 file_num
wav_list = list(ORIGINAL.glob('*.wav'))
print(f'找到 {len(wav_list)} 个 wav')

def parse_num(name): 
    m = re.search(r'(\d{2})\.wav$', name)
    return int(m.group(1)) if m else -1

file_meta = []
for p in wav_list:
    num = parse_num(p.name)
    if num == -1: continue
    dur = sf.info(p).duration
    file_meta.append({'file': p.name, 'file_num': num, 'duration': dur, 'path': p})

meta = pd.DataFrame(file_meta).sort_values('file_num')
meta.to_csv(ROOT/'file_meta.csv', index=False)
print(meta[['file_num', 'duration']].head())

找到 35 个 wav
   file_num     duration
0         1  1799.440528
1         2  1799.953556
2         3  1799.969222
3         4  1799.931861
4         5  1799.946167


In [8]:
# 3. 下采样 + 切割 1min bag + 打弱标签 + 保存
bag_records = []

for _, row in tqdm(meta.iterrows(), total=len(meta), desc='切割音频'):
    fnum = row['file_num']
    audio, sr = librosa.load(row['path'], sr=48000)   # 直接下采样到48kHz
    bag_len = 48000 * 60
    n_bags = len(audio) // bag_len
    
    file_trains = trains[trains['file_num'] == fnum]
    
    out_dir = ROOT / 'dataset' / 'bags' / f'file_{fnum:02d}'
    out_dir.mkdir(exist_ok=True)
    
    for k in range(n_bags):
        start_s = k * 60.0
        seg = audio[k*bag_len : (k+1)*bag_len]
        bag_path = out_dir / f'bag_{k:03d}.wav'
        sf.write(bag_path, seg, 48000)
        
        label = 0
        for _, t in file_trains.iterrows():
            if max(start_s, t['start']) < min(start_s+60, t['end']):
                label = 1
                break
        
        bag_records.append({
            'file_num': fnum,
            'bag_idx': k,
            'label': label,
            'bag_path': str(bag_path.relative_to(ROOT)),
            'start_sec': start_s
        })

bag_df = pd.DataFrame(bag_records)
bag_df.to_csv(ROOT / 'all_bags_48kHz.csv', index=False)
print(f'✅ 已保存 {len(bag_df)} 个 1min bag（48kHz）')

切割音频: 100%|██████████| 35/35 [03:07<00:00,  5.35s/it]

✅ 已保存 986 个 1min bag（48kHz）


In [9]:
# 4. 5折验证统计（按时间顺序）
stats = []
for fnum in bag_df['file_num'].unique():
    sub = bag_df[bag_df['file_num'] == fnum]
    labels = sub['label'].values
    n = len(labels)
    if n < 5: continue
    
    kf = KFold(n_splits=5, shuffle=False)  # 时间顺序 8:2
    for fold, (_, test_idx) in enumerate(kf.split(range(n))):
        test_lab = labels[test_idx]
        pos = (test_lab == 1).sum()
        neg = (test_lab == 0).sum()
        stats.append({
            'fold': fold + 1,
            'file_num': fnum,
            'test_total_bags': len(test_idx),
            'test_pos_1': int(pos),
            'test_neg_0': int(neg),
            'pos_ratio': round(pos/len(test_idx), 3)
        })

stats_df = pd.DataFrame(stats)
stats_df.to_csv(ROOT / 'bag_test_stats_5fold_detailed.csv', index=False, encoding='utf-8-sig')

# 美观展示 + 汇总
summary = stats_df.groupby('file_num').agg({
    'test_pos_1': ['mean', 'sum'],
    'test_neg_0': ['mean', 'sum'],
    'test_total_bags': 'mean'
}).round(2)
summary.columns = ['avg_pos_per_fold', 'total_pos_across_5folds', 'avg_neg_per_fold', 'total_neg_across_5folds', 'avg_test_bags_per_fold']
display(summary)
print('📊 完整统计已保存 → bag_test_stats_5fold_detailed.csv')

,avg_pos_per_fold,total_pos_across_5folds,avg_neg_per_fold,total_neg_across_5folds,avg_test_bags_per_fold
file_num,,,,,
1,1.2,6,4.6,23,5.8
2,4.2,21,1.6,8,5.8
3,1.0,5,4.8,24,5.8
4,1.2,6,4.6,23,5.8
5,2.0,10,3.8,19,5.8
6,3.0,15,2.8,14,5.8
7,1.2,6,4.6,23,5.8
8,2.4,12,3.4,17,5.8
9,3.8,19,2.0,10,5.8


📊 完整统计已保存 → bag_test_stats_5fold_detailed.csv


**运行完毕！**

✅ 根目录现在有：
- `all_bags_48kHz.csv`（所有bag路径+标签）
- `bag_test_stats_5fold_detailed.csv`（你要求的「5次分割中，每个.wav文件的测试集正负bag数量」）
- `dataset/bags/file_01/` … `file_35/`（已切好的48kHz 1min wav，可直接喂MIL）

后续MIL训练只需读取 `all_bags_48kHz.csv` 并按fold筛选即可。

需要我再加「生成 .npz 特征文件」「把bag按fold复制到train/test文件夹」或其他部分，直接告诉我！🚀